# Heat Pump Design

This notebook explores a minimal heat pump model for Utrecht with annual demand, seasonal COP, monthly and daily distribution, and simple cost/CO2 outputs.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from home_energy.domain import HeatPumpConfig
from home_energy.heat_pump import (
    build_heat_pump_daily_profile_figure,
    build_heat_pump_energy_source_breakdown_figure,
    build_heat_pump_monthly_figure,
    heat_pump_explainability_summary,
    heat_pump_validation_warnings,
    simulate_heat_pump,
    simulate_heat_pump_interaction,
)

In [ ]:
config = HeatPumpConfig(
    enabled=True,
    annual_heat_demand_kwh=8000.0,
    seasonal_cop=4.0,
    hot_water_included=True,
)

result = simulate_heat_pump(config, year=2026)
result.summary_rows()

In [ ]:
warnings = heat_pump_validation_warnings(config, household_consumption_kwh=3500.0)
for warning in warnings:
    print(f'WARNING: {warning}')

In [ ]:
monthly_figure = build_heat_pump_monthly_figure(result)
monthly_figure

In [ ]:
hourly_price = [0.22 if 11 <= hour <= 15 else 0.34 for hour in range(24)]
dynamic_price = [hourly_price[ts.hour] for ts in result.timestamps]

interaction = simulate_heat_pump_interaction(
    result,
    dynamic_price_eur_per_kwh=dynamic_price,
    grid_co2_kg_per_kwh=0.35,
)

source_breakdown_figure = build_heat_pump_energy_source_breakdown_figure(interaction)
daily_profile_figure = build_heat_pump_daily_profile_figure(interaction)
interaction.summary_rows()

In [ ]:
source_breakdown_figure
daily_profile_figure
heat_pump_explainability_summary(interaction)